# Gold — Turismo Bilateral Brasil–Coreia do Sul

## 1. Construção da tabela analítica anual

Objetivo: construir uma tabela comparativa das chegadas
internacionais entre Brasil e Coreia do Sul.

Período pretendido: 1990–2025.

Fontes:
- chegadas.silver.brasil
- chegadas.silver.coreia

Regras:
- Selecionar os registros relevantes para cada fluxo.
- Preservar as tabelas Silver originais.
- Não misturar totais anuais e registros mensais.
- Verificar a integridade dos indicadores.
- Documentar as diferenças metodológicas entre as fontes.

In [0]:
from pyspark.sql import functions as F

df_brasil = spark.table("chegadas.silver.brasil")
df_coreia = spark.table("chegadas.silver.coreia")

In [0]:
df_coreanos_brasil = (
    df_brasil
    .filter(
        F.col("pais") == "República da Coreia"
    )
)

display(
    df_coreanos_brasil
    .select(
        "pais",
        "ano",
        "mes",
        "uf",
        "via",
        "chegadas"
    )
    .orderBy("ano", "mes")
    .limit(20)
)

print(
    "Registros selecionados:",
    df_coreanos_brasil.count()
)

In [0]:
df_brasileiros_coreia = (
    df_coreia
    .filter(
        (F.col("categoria_en") == "Brazil")
))

display(
    df_brasileiros_coreia
    .select(
        "categoria_en",
        "categoria_ko",
        "ano",
        "mes",
        "granularidade",
        "chegadas_brutas",
        "chegadas"
    )
    .orderBy("ano")
)

print(
    "Registros anuais selecionados:",
    df_brasileiros_coreia.count()
)

In [0]:
display(
    df_coreanos_brasil.agg(
        F.count("*").alias("total_registros"),
        F.min("ano").alias("ano_inicial"),
        F.max("ano").alias("ano_final"),
        F.sum(
            F.when(
                F.col("chegadas").isNull(), 1
            ).otherwise(0)
        ).alias("chegadas_nulas")
    )
)

display(
    df_brasileiros_coreia.agg(
        F.count("*").alias("total_registros"),
        F.min("ano").alias("ano_inicial"),
        F.max("ano").alias("ano_final"),
        F.sum(
            F.when(
                F.col("chegadas").isNull(), 1
            ).otherwise(0)
        ).alias("chegadas_nulas")
    )
)

Dados anuais

In [0]:
display(
    df_coreanos_brasil
    .filter(F.col("chegadas").isNull())
    .groupBy("ano")
    .count()
    .orderBy("ano")
)

In [0]:
df_brasil_anual = (
    df_coreanos_brasil
    .groupBy("ano")
    .agg(
        F.sum("chegadas").alias("chegadas_observadas"),

        F.count("*").alias("total_registros"),

        F.sum(
            F.when(
                F.col("chegadas").isNull(), 1
            ).otherwise(0)
        ).alias("registros_nulos")
    )
    .withColumn(
        "chegadas",
        F.when(
            F.col("registros_nulos") == 0,
            F.col("chegadas_observadas")
        ).otherwise(F.lit(None).cast("bigint"))
    )
    .withColumn(
        "status_dados",
        F.when(
            F.col("registros_nulos") == 0,
            F.lit("sem_nulos")
        ).otherwise(F.lit("incompleto"))
    )
    .orderBy("ano")
)

display(df_brasil_anual)

In [0]:
df_coreia_anual = (
    spark.table("chegadas.silver.coreia")

    .filter(
        (F.col("categoria_en") == "Brazil") &
        (F.col("granularidade") == "anual") &
        (F.col("mes").isNull()) &
        (F.col("ano").between(1990, 2025))
    )

    .select(
        "ano",
        F.col("chegadas").cast("bigint").alias("chegadas")
    )

    .withColumn(
        "chegadas_observadas",
        F.col("chegadas")
    )

    .withColumn("total_registros", F.lit(1))

    .withColumn(
        "registros_nulos",
        F.when(
            F.col("chegadas").isNull(), 1
        ).otherwise(0)
    )

    .withColumn(
        "status_dados",
        F.when(
            F.col("chegadas").isNull(),
            F.lit("incompleto")
        ).otherwise(F.lit("sem_nulos"))
    )
)

display(df_coreia_anual.orderBy("ano"))

print("Registros anuais:", df_coreia_anual.count())

In [0]:
# Fluxo 1: Coreia do Sul → Brasil

df_fluxo_brasil = (
    df_brasil_anual
    .withColumn("origem", F.lit("Coreia do Sul"))
    .withColumn("destino", F.lit("Brasil"))
)

# Fluxo 2: Brasil → Coreia do Sul

df_fluxo_coreia = (
    df_coreia_anual
    .withColumn("origem", F.lit("Brasil"))
    .withColumn("destino", F.lit("Coreia do Sul"))
)

# União dos dois fluxos

df_gold_anual = (
    df_fluxo_brasil
    .unionByName(df_fluxo_coreia)
    .select(
        "ano",
        "origem",
        "destino",
        "chegadas",
        "chegadas_observadas",
        "total_registros",
        "registros_nulos",
        "status_dados"
    )
    .orderBy("ano", "origem")
)

display(df_gold_anual)

In [0]:
# Fluxo Coreia do Sul → Brasil
df_fluxo_brasil = (
    df_brasil_anual
    .withColumn("origem", F.lit("Coreia do Sul"))
    .withColumn("destino", F.lit("Brasil"))
)

# Fluxo Brasil → Coreia do Sul
df_fluxo_coreia = (
    df_coreia_anual
    .withColumn("origem", F.lit("Brasil"))
    .withColumn("destino", F.lit("Coreia do Sul"))
)

# Reconstrução da Gold anual
df_gold_anual = (
    df_fluxo_brasil
    .unionByName(df_fluxo_coreia)
    .select(
        "ano",
        "origem",
        "destino",
        "chegadas",
        "chegadas_observadas",
        "total_registros",
        "registros_nulos",
        "status_dados"
    )
    .orderBy("ano", "origem")
)

In [0]:
print("Total de registros:", df_gold_anual.count())

display(
    df_gold_anual
    .groupBy("ano", "origem", "destino")
    .count()
    .filter(F.col("count") != 1)
)

display(
    df_gold_anual
    .filter(F.col("ano") == 2025)
)

1° Tabela Gold

In [0]:
# Criar o schema Gold
spark.sql("""CREATE SCHEMA IF NOT EXISTS chegadas.gold""")

# Definir o nome da tabela
TABELA_GOLD = "chegadas.gold.turismo_bilateral_anual"

# Salvar a tabela analítica
(
    df_gold_anual.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA_GOLD)
)

print("Tabela Gold anual criada!")

In [0]:
df_gold_salva = spark.table(TABELA_GOLD)

print("Total de registros:", df_gold_salva.count())

display(
    df_gold_salva
    .groupBy("ano", "origem", "destino")
    .count()
    .filter(F.col("count") != 1)
)

assert df_gold_salva.count() == 72

print("Validação da tabela anual concluída!")